# ERA5 $U_{speed}$ climatology & $Sea Level Pressure$ climatology

In [32]:
import os
import numpy as np
import scipy as sp
import pandas as pd
from datetime import date
import marineHeatWaves as mhw
import netCDF4 as nc
import datetime
import matplotlib.pyplot as plt
from tqdm import notebook
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, Executor
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.backends.backend_agg import FigureCanvasAgg
import PIL.Image as Image
from scipy.stats import pearsonr
import matplotlib

In [33]:
Eastfile='/lustre/home/yuhanxue/data/ERA5_monthly_slp&Windspeed/120E_180.nc'
Westfile='/lustre/home/yuhanxue/data/ERA5_monthly_slp&Windspeed/180_90W.nc'
daset1=nc.Dataset(Eastfile)
daset2=nc.Dataset(Westfile)

In [34]:
def getallname(s):
    sl=s.split()
    rs=''
    for i in sl:
        rs+=i
        rs+='_'
    return rs[:-1]


names=[]
for i in daset1.variables.keys():
    print(f'{i}:{getallname(daset1.variables[i].long_name)} ({daset1.variables[i].units})')
    names.append(getallname(daset1.variables[i].long_name))
    #print(np.array(daset1.variables[i]).shape)

longitude:longitude (degrees_east)
latitude:latitude (degrees_north)
time:time (hours since 1900-01-01 00:00:00.0)
si10:10_metre_wind_speed (m s**-1)
msl:Mean_sea_level_pressure (Pa)


In [40]:
time

DatetimeIndex(['1993-01-01', '1993-02-01', '1993-03-01', '1993-04-01',
               '1993-05-01', '1993-06-01', '1993-07-01', '1993-08-01',
               '1993-09-01', '1993-10-01',
               ...
               '2021-03-01', '2021-04-01', '2021-05-01', '2021-06-01',
               '2021-07-01', '2021-08-01', '2021-09-01', '2021-10-01',
               '2021-11-01', '2021-12-01'],
              dtype='datetime64[ns]', length=348, freq=None)

In [35]:
lon_E=np.array(daset1['longitude'])
lon_W=np.array(daset2['longitude'])+360
lon=np.concatenate([lon_E,lon_W])
lon=np.delete(lon,241)
lat=np.array(daset1['latitude'])

In [36]:
def cftime2pdtime(cf):
    return pd.to_datetime(datetime.datetime(cf.year,cf.month,cf.day,cf.hour))
time=pd.to_datetime(list(map(cftime2pdtime,nc.num2date(np.array(daset1.variables['time']),daset1.variables['time'].units))))

In [37]:
si10_E=np.array(daset1['si10'])
msl_E=np.array(daset1['msl'])
si10_W=np.array(daset2['si10'])
msl_W=np.array(daset2['msl'])
si10=np.delete(np.concatenate([si10_E,si10_W],axis=2),241,axis=2)
msl=np.delete(np.concatenate([msl_E,msl_W],axis=2),241,axis=2)

In [38]:
timind=[time.month==(i+1) for i in range(12)]

In [39]:
si10_clim=np.copy(si10)
msl_clim=np.copy(msl)
for i in range(12):
    si10_clim[timind[i],:,:]=np.nanmean(si10_clim[timind[i],:,:],axis=0)
    msl_clim[timind[i],:,:]=np.nanmean(msl_clim[timind[i],:,:],axis=0)

In [52]:
intro="time:Monthly,1993-1~2021~12 \nsi10_clim:10_metre_wind_speed_climatology(m s**-1)\nsi10_ano:10_metre_wind_speed_anomaly(m s**-1)\n"+\
"msl_clim:Mean_sea_level_pressure_climatology(Pa)\nmsl_ano:Mean_sea_level_pressure_anomaly(Pa)"+\
" \nlon:longitude(degrees_east) \nlat:latitude(degrees_north)"

In [53]:
print(intro)

time:Monthly,1993-1~2021~12 
si10_clim:10_metre_wind_speed_climatology(m s**-1)
si10_ano:10_metre_wind_speed_anomaly(m s**-1)
msl_clim:Mean_sea_level_pressure_climatology(Pa)
msl_ano:Mean_sea_level_pressure_anomaly(Pa) 
lon:longitude(degrees_east) 
lat:latitude(degrees_north)


In [57]:
np.unique(si10_clim)

array([ 0.69759078,  0.71660141,  0.74019011, ..., 12.46734502,
       12.60801805, 12.7087551 ])

In [54]:
np.savez("12_13_ERA5_u_slp.npz",intro=intro,time=time,si10_clim=np.unique(si10_clim),si10_ano=si10-si10_clim,msl_clim=msl_clim,msl_ano=msl-msl_clim,lon=lon,lat=lat)

In [55]:
import scipy.io as scio

In [56]:
scio.savemat("12_13_ERA5_u_slp.mat",{"intro":intro,"time":time,"si10_clim":si10_clim,"si10_ano":si10-si10_clim,"msl_clim":msl_clim,"msl_ano":msl-msl_clim,"lon":lon,"lat":lat})